In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, accuracy_score
import os
from tensorflow.keras import layers, applications

In [4]:
preprocess_fn = tf.keras.applications.mobilenet_v2.preprocess_input

TRAIN_DIR = '/kaggle/input/5-fold-brain-tumor-contrast-enhanced/kfold_dataset'

In [5]:
def build_data_augmentation(SEED=24520152):
    """Create a simple data augmentation pipeline"""
    return tf.keras.Sequential([
        layers.RandomFlip('horizontal', seed=SEED),
        layers.RandomRotation(0.1, seed=SEED),
        layers.RandomZoom(0.1, seed=SEED),
        layers.RandomContrast(0.1, seed=SEED),
        layers.RandomBrightness(0.1, seed=SEED),
    ], name='data_augmentation')

In [6]:
def get_data_for_fold(k, preprocess_fn, train_dir=TRAIN_DIR, IMG_SIZE=(224, 224), BATCH_SIZE=32, SEED=24520152):
    val_dir = os.path.join(train_dir, f'Subset_{k}')
    train_dirs = [os.path.join(train_dir, f'Subset_{i}') for i in range(1, 6) if i != k]
    
    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
        seed=SEED,
    )
    
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dirs[0],
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=SEED,
    )
    
    for directory in train_dirs[1:]:
        ds_part = tf.keras.utils.image_dataset_from_directory(
            directory,
            image_size=IMG_SIZE,
            batch_size=BATCH_SIZE,
            shuffle=True,
            seed=SEED,
        )
        train_ds = train_ds.concatenate(ds_part)

    train_ds = train_ds.shuffle(buffer_size=3000, seed=SEED)
    
    AUTOTUNE = tf.data.AUTOTUNE
    data_augmentation = build_data_augmentation()

    def augment_and_preprocess_train(image, label):
        """Apply augmentation and preprocessing to the training set"""
        image = data_augmentation(image, training=True)
        image = preprocess_fn(image)
        return image, label

    def preprocess_val(image, label):
        """Apply preprocessing to the validation set (no augmentation)"""
        image = preprocess_fn(image)
        return image, label

    train_ds = train_ds.map(augment_and_preprocess_train, num_parallel_calls=AUTOTUNE).prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.map(preprocess_val, num_parallel_calls=AUTOTUNE).prefetch(buffer_size=AUTOTUNE)

    return train_ds, val_ds

In [7]:
def get_model_for_fold(i, k, input_dir):
    model = tf.keras.models.load_model(f'{input_dir}/Fold_{k}/MobileNetV2_block_{i}_fold_{k}.keras', compile=False)
    return model

DATASET_CACHE = {}
def get_valid_data_for_fold(k):
    if k in DATASET_CACHE:
        return DATASET_CACHE[k]
        
    _, val_ds = get_data_for_fold(k, preprocess_fn)

    _ = _.cache().prefetch(tf.data.AUTOTUNE)
    val_ds   = val_ds.cache().prefetch(tf.data.AUTOTUNE)

    DATASET_CACHE[k] = val_ds
    return val_ds

In [8]:
def get_predictions_for_fold(k, input_dir):
    valid_data = get_valid_data_for_fold(k)

    y_true = []
    for _, y in valid_data:
        y_true.extend(y.numpy())
    y_true = np.array(y_true)
    y_pred = []

    for i in range(1, 6):
        model = get_model_for_fold(i, k, input_dir)
        preds = model.predict(valid_data, verbose=0)
        y_pred.append(preds)

    y_pred = np.array(y_pred)
    return y_true, y_pred

In [9]:
def macro_specificity(y_true, y_pred, num_classes):
    cm = confusion_matrix(y_true, y_pred)
    spec = []

    for i in range(num_classes):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - (TP + FN + FP)

        spec.append(TN / (TN + FP + 1e-8))

    return np.array(spec)

In [10]:
def evaluate_fold(k, input_dir):
    y_true, y_pred = get_predictions_for_fold(k, input_dir)

    model_results = []
    for i in range(len(y_pred)):
        y_pred_label = np.argmax(y_pred[i], axis=1)
        acc = accuracy_score(y_true, y_pred_label)
        precision = precision_score(y_true, y_pred_label, average='macro', zero_division=0)
        recall = recall_score(y_true, y_pred_label, average='macro', zero_division=0)
        f1 = f1_score(y_true, y_pred_label, average='macro', zero_division=0)
        spec_per_class = macro_specificity(y_true, y_pred_label, 3)
        specificity = np.mean(spec_per_class)

        model_results.append({
            'Accuracy': np.array(acc),
            'Precision': np.array(precision),
            'Recall': np.array(recall),
            'F1-Score': np.array(f1),
            'Specificity': np.array(specificity)
        })
    return np.array(model_results)

In [11]:
input_dir = '/kaggle/input/mobilenetv2-brain-tumor-model/MobileNetV2'

five_fold_results = []
for i in range(1, 6):
    five_fold_results.append(evaluate_fold(i, input_dir))

five_fold_results = np.array(five_fold_results)

num_fold = 5
num_model = 5
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Specificity']

avg_metrics = {}

for m in range(num_model):
    avg_metrics[m] = {}
    for metric in metric_names:
        values = np.array([
            five_fold_results[k][m][metric] for k in range(num_fold)
        ])
        avg_metrics[m][metric] = values.mean(axis=0)*100

Found 542 files belonging to 3 classes.


I0000 00:00:1767453106.971446      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Found 679 files belonging to 3 classes.
Found 572 files belonging to 3 classes.
Found 628 files belonging to 3 classes.
Found 643 files belonging to 3 classes.


I0000 00:00:1767453114.913528     133 service.cc:152] XLA service 0x7f9948110a90 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1767453114.913563     133 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1767453115.536738     133 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1767453120.874157     133 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Found 679 files belonging to 3 classes.
Found 542 files belonging to 3 classes.
Found 572 files belonging to 3 classes.
Found 628 files belonging to 3 classes.
Found 643 files belonging to 3 classes.
Found 572 files belonging to 3 classes.
Found 542 files belonging to 3 classes.
Found 679 files belonging to 3 classes.
Found 628 files belonging to 3 classes.
Found 643 files belonging to 3 classes.
Found 628 files belonging to 3 classes.
Found 542 files belonging to 3 classes.
Found 679 files belonging to 3 classes.
Found 572 files belonging to 3 classes.
Found 643 files belonging to 3 classes.
Found 643 files belonging to 3 classes.
Found 542 files belonging to 3 classes.
Found 679 files belonging to 3 classes.
Found 572 files belonging to 3 classes.
Found 628 files belonging to 3 classes.


In [12]:
row_names = ['FT: B$_1$-B$_5$', 'FT: B$_2$-B$_5$', 'FT: B$_3$-B$_5$', 'FT: B$_4$-B$_5$', 'FT: B$_5$']
column_names = ['Recall', 'Specificity', 'Precision', 'F1-Score', 'Accuracy']

df_result = pd.DataFrame(avg_metrics).T
df_result.index = row_names
df_result.index.name = 'Fine-tuning'
df_result = df_result[column_names]
df_result = df_result.round(2)
df_result

,Recall,Specificity,Precision,F1-Score,Accuracy
Fine-tuning,,,,,
FT: B$_1$-B$_5$,88.92,94.71,87.16,87.63,88.70
FT: B$_2$-B$_5$,88.92,94.86,87.89,88.20,89.44
FT: B$_3$-B$_5$,87.60,94.25,87.04,87.12,88.32
FT: B$_4$-B$_5$,85.92,93.70,86.26,85.84,87.68
FT: B$_5$,85.95,93.43,84.68,85.06,86.47


In [13]:
latex_table = df_result.to_latex(
    multicolumn=True,
    multirow=True,
    float_format="%.2f",
    label="tab:sens_spec"
)

print(latex_table)

\begin{table}
\label{tab:sens_spec}
\begin{tabular}{lrrrrr}
\toprule
 & Recall & Specificity & Precision & F1-Score & Accuracy \\
Fine-tuning &  &  &  &  &  \\
\midrule
FT: B$_1$-B$_5$ & 88.92 & 94.71 & 87.16 & 87.63 & 88.70 \\
FT: B$_2$-B$_5$ & 88.92 & 94.86 & 87.89 & 88.20 & 89.44 \\
FT: B$_3$-B$_5$ & 87.60 & 94.25 & 87.04 & 87.12 & 88.32 \\
FT: B$_4$-B$_5$ & 85.92 & 93.70 & 86.26 & 85.84 & 87.68 \\
FT: B$_5$ & 85.95 & 93.43 & 84.68 & 85.06 & 86.47 \\
\bottomrule
\end{tabular}
\end{table}

